 # RSA



 **Reminder**: (Fermat's little theorem):

 $$ a^{p-1} \equiv \left\{ \begin{array}{rcl}

 1 \pmod{p}, & \text{if} & p\nmid a \\

 0 \pmod{p}, & \text{if} & p \mid a \\

 \end{array}\right.$$



 Is Fermat's little theorem still true if we replace $p$ with a number that is not a prime? Let $m=15485207$:

In [0]:
m = 15485207
mod(2 ^ (m - 1), m)


 But Fermat's little theorem can be generalized. Let $m = 15$, then

 $$

 \begin{array}{ll}

     a^4 \equiv 1 \pmod{15} & \text{if}\; a=1,2,4,7,8,11,13,14 \\

     a^4 \not\equiv 1 \pmod{15} & \text{if}\; a=3,5,6,9,10,12 \\

 \end{array}

 $$

 **Question**: What distinguishes the numbers $a=1,2,4,7,8,11,13,14$ from the numbers $a=3,5,6,9,10,12$?

 **Solution**: The elements in the first list are relatively prime to $15$, while the prime factorization of the elements in the second list contains factors that are also present in the factorization of $15$. That is, if $a$ and the modulus $m$ are relatively prime, then some form of Fermat's little theorem must hold. But in this case, the correct exponent is not necessarily $m - 1$.

 Why is the exponent $4$ a good choice? Let's look at the following congruences:

 $$

 a^4 \equiv 1 \pmod{3}, \quad a^4 \equiv 1 \pmod{5}

 $$

 From these two congruences, we get that $a^4 \equiv 1 \pmod{15}$. (The two congruences are true, which follows from Fermat's little theorem)



 Another observation: the exponent $4$ is a multiple of $p-1$ for $p=3$ and $p=5$ *as well*.



 **Theorem** (Euler's formula for $pq$): Let $p,q$ be distinct prime numbers and $g = \text{gcd}(p-1, q-1)$. Then $$a^{(p-1)(q-1)/g} \equiv 1 \pmod{pq}$$ for all $a$ such that $\text{gcd}(a,pq)=1$. If $p,q$ are odd primes, then $$a^{(p-1)(q-1)/2} \equiv 1 \pmod{pq}$$ for all $a$ such that $\text{gcd}(a,pq)=1$.

 Diffie-Hellman: Relies on the difficulty of $a^x \equiv b \pmod{p}$



 RSA: Relies on the difficulty of computing $x^e \equiv c \pmod{N}$ (i.e., computing the $e$-th power modulo $N$ is difficult). How realistic is this if $N$ is a prime number?

 **Theorem**: Let $p$ be a prime number and $e \geq 1$ such that $\text{gcd}(e, p-1) = 1$. Then $e$ has a multiplicative inverse modulo $p - 1$, $d$, that is, $de \equiv 1 \pmod{p-1}$. Then the congruence $x^e \equiv c \pmod{p}$ has the solution $x \equiv c^d \pmod{p}$ and $x$ is unique.

 **Example**: Let $m = 7919, e = 1583, c = 4714$. Solve the congruence $x^{1583} \equiv 4714 \pmod{7919}$!

In [0]:
m = 7919
e = 1583
c = 4714


In [0]:
d = inverse_mod(e, m - 1)
d


In [0]:
mod(c ^ d, m)


In [0]:
mod(6059 ^ e, m) == c


 If $N=pq$, the situation is different:



 **Theorem**: Let $p,q$ be prime numbers and $e \geq 1$ such that $\text{gcd}(e, (p-1)(q-1)) = 1$. Then $e$ has a multiplicative inverse modulo $(p-1)(q-1)$, that is, $de \equiv 1 \pmod{(p-1)(q-1)}$. Then the congruence $x^e \equiv c \pmod{pq}$ has the solution $x \equiv c^d \pmod{pq}$ and $x is unique.



 **Remark**: If the factorization of $N$ is not known, the above theorem cannot be used to solve the congruence.

 **Example**: Let $N = 30069476293, e = 9843, c = 134872$. Solve the congruence $x^{9843} \equiv 1348772 \pmod{30069476293}$!

In [0]:
N = 30069476293
z = Integers(N)
e = z(9843)
c = z(134872)


In [0]:
f = factor(N)
p, q = f[0][0], f[1][0]
p, q


In [0]:
d = inverse_mod(int(e), (p - 1) * (q - 1))
d


In [0]:
c ^ z(d)


 ## (Textbook) RSA (Rivest-Shamir-Adleman)



 Key Generation:

 - Alice chooses two primes, which she keeps secret, but makes $N=pq$ public



 Encryption:

 - Alice chooses a *public* exponent $e$ such that $\text{gcd}(e, (p-1)(q-1))=1$

 - The message $m$ to be encrypted is an integer $1 \leq m < N$

 - Bob encrypts the message $m$ using $e,N$: $c \equiv m^e \pmod{N}$

 - Bob sends the encrypted message to Alice



 Decryption:

 - Alice knows the values of $p,q$, so she can compute the multiplicative inverse of $e$: $de \equiv 1 \pmod{(p-1)(q-1)}$

 - Using the value of $d$, Alice can decrypt the message: $m \equiv c^d \pmod{N}$

 **Remarks**:

 - We can also choose $e=3$, *in principle* this is just as secure as a larger value ($e=1$ why not good?)

 - A common choice is $e=2^{16}+1 = 65537$ - this keeps the encryption fast (4 exponentiations and 1 multiplication to compute $m^e \mod{N}$)

 - But $d > N^{1/4}$ must hold

 An attack:

 - Eve asks Alice to decrypt a message, Alice happily does (thus authenticating herself)

 - Eve intercepts Bob's ciphertext $c$.

 - Eve chooses a random $k$ and forwards the ciphertext $c' \equiv k^e \cdot c \pmod{N}$ to Alice

 - Alice decrypts $c'$ and sends Eve: $m' \equiv (c')^d \equiv (k^e \cdot c)^d \equiv (k^e \cdot m^e)^d \equiv k \cdot m \pmod{N}$

 - Eve now knows $k \cdot m \pmod{N}$, and since she chose $k$, she can recover Bob's message $m$



 Remarks:

 1. Eve has decrypted Bob's message *without* factoring $N$

 2. Alice will not realize that Eve is an attacker, as $k^e \cdot c \pmod{N}$ and $k \cdot m \pmod{N}$ appear random



 Here: We choose only one $e$ exponent for a given $N$:

 - Choose $e_1,e_2$ exponents

 - Bob encrypts a message $m$ with the two exponents: $c_1 \equiv m^{e_1} \pmod{N}$ and $c_2 \equiv m^{e_2} \pmod{N}$

 - Eve intercepts this, from which she gets a solution $e_1 \cdot u + e_2 \cdot v = \text{gcd}(e_1, e_2)$

 - From this, she can decrypt the message if $\text{gcd}(e_1, e_2) = 1$: $c_1^u \cdot c_2^v \equiv (m_1^e)^u \cdot (m_2^e)^v \equiv m^{e_1 \cdot u + e_2 \cdot v} \equiv m^{\text{gcd}(e_1,e_2)} \pmod{N}$

 **Important Security Notes**:

 1. The security of RSA relies on the fact that factoring large composite numbers is hard. However, some numbers are easier to factor:

    - If p and q are too close together

    - If p-1 and q-1 have only small prime factors

    - If N is too small (current recommendation is at least 2048 bits)



 2. Side-channel attacks are possible through:

    - Timing measurements of modular exponentiation

    - Power analysis during decryption

    - Cache behavior analysis



 3. Implementations should use CRT (Chinese Remainder Theorem) for decryption:

    - Compute mp = c^d mod p

    - Compute mq = c^d mod q

    - Combine using CRT to get m

    This is about 4 times faster than direct computation.

 ## RSA in practice



 The [RFC 8017](https://datatracker.ietf.org/doc/html/rfc8017) standard specifies two constructions for using RSA:

 - RSA OAEP (`RSAES-OAEP`)

 - Padded RSA (`RSAES-PKCS1-v1_5`)

 ### RSA OAEP



 OAEP: Optimal Asymmetric Encryption scheme (Bellare and Rogaway, 1995, 1998)



 ![OAEP encoding](./oaep_encoding.png)



 Encryption: Encode the plaintext and then apply textbook RSA encryption



 Decryption: Apply textbook RSA decryption and then decode the result

 ### OAEP Implementation Details

 OAEP uses two hash functions (typically SHA-256):

 - MGF (Mask Generation Function): Generates a mask of arbitrary length to be XORed with the plaintext

 - Hash function H: Creates a fixed-length digest



 The padding scheme provides:

 1. Semantic security - same message encrypted twice produces different ciphertexts

 2. Non-malleability - prevents multiplication attacks shown earlier

 3. All-or-nothing security - partial decryption is impossible

 ### Padded RSA



 EM = 0x00 || 0x02 || PS || 0x00 || M



 - EM: Encoded message

 - PS: Padding string of random bytes

 - M: Message



 The padding string PS must be at least 8 bytes long. The message M must be less than the modulus N.



 The padding scheme provides:

 1. Semantic security - same message encrypted twice produces different ciphertexts

 2. Non-malleability - prevents multiplication attacks shown earlier

 3. All-or-nothing security - partial decryption is impossible

